# 03. Evaluación de la línea base de persistencia

Este cuaderno muestra cómo se comportó la persistencia en junio. La predicción repite la medición real de 24 horas antes y se evalúan solo pares observados.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

carpeta = Path.cwd()
if not (carpeta / "data").exists():
    carpeta = carpeta.parent
datos = pd.read_csv(carpeta / "data" / "particion temporal.csv", parse_dates=["fecha_objetivo"])
with open(carpeta / "experiments" / "resultados linea de base.json", encoding="utf-8") as archivo:
    resultados = json.load(archivo)
prueba = datos[datos.particion == "test"].dropna(subset=["pm25_original", "target_24h"]).copy()
prueba["prediccion_persistencia"] = prueba.pm25_original
prueba["error"] = prueba.target_24h - prueba.prediccion_persistencia
mae = prueba.error.abs().mean()
rmse = (prueba.error.pow(2).mean()) ** 0.5
assert len(prueba) == resultados["muestras_evaluadas"]
assert round(mae, 2) == resultados["mae_ug_m3"]
assert round(rmse, 2) == resultados["rmse_ug_m3"]
print(f"Pares: {len(prueba)} | MAE: {mae:.2f} | RMSE: {rmse:.2f} µg/m³")

## Valor real y predicción

Las líneas se separan cuando la concentración cambia entre una hora y la misma hora del día siguiente.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(prueba.fecha_objetivo, prueba.target_24h, label="Real", linewidth=1, color="black")
ax.plot(prueba.fecha_objetivo, prueba.prediccion_persistencia, label="Persistencia", linewidth=1, color="royalblue", alpha=0.8)
ax.set(title="Junio: valor real y predicción de persistencia", xlabel="Fecha objetivo", ylabel="PM2.5 (µg/m³)")
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## Error por hora

El residuo es valor real menos predicción. Un valor positivo indica que la persistencia subestimó PM2.5.

In [ ]:
fig, ax = plt.subplots(figsize=(13, 3.5))
ax.axhline(0, color="black", linewidth=1)
ax.bar(prueba.fecha_objetivo, prueba.error, width=0.025, color="darkorange")
ax.set(title="Error de persistencia durante junio", xlabel="Fecha objetivo", ylabel="Error (µg/m³)")
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

## Distribución del error

El MAE es la magnitud absoluta promedio, no el centro del histograma. Se marcan cero y ±MAE como referencia.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(prueba.error, bins=30, color="mediumpurple", edgecolor="white")
ax.axvline(0, color="black", label="Error cero")
ax.axvline(mae, color="darkorange", linestyle="--", label=f"+MAE ({mae:.2f})")
ax.axvline(-mae, color="darkorange", linestyle="--", label=f"−MAE ({mae:.2f})")
ax.set(title="Distribución de errores", xlabel="Real − predicción (µg/m³)", ylabel="Número de horas")
ax.legend()
ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()